# 06 — Qué sale mal

**Bloque 5, 30 min. PENDIENTE del caso de estudio.**

**La parte que separa este taller de un tutorial.** Cinco demos preparadas,
3-4 minutos cada una. Todas se corren en vivo y todas fallan de verdad.

| # | Falla | Qué se ve |
|---|---|---|
| 1 | **Bucle infinito** | El modelo consulta lo mismo una y otra vez. El tope de iteraciones salvando la situación. |
| 2 | **Argumento alucinado** | Inventa una clave que no existe. El error volviendo al modelo, y el modelo corrigiéndose — o no. |
| 3 | **Teléfono descompuesto** | Recepción resume mal y Pedidos ejecuta lo equivocado. El costo de que el contexto no viaje completo. |
| 4 | **Destructivo sin red** | Se apaga la confirmación y se cancela lo que no debía. |
| 5 | **El costo escondido** | Contar las llamadas al modelo del multiagente contra el agente único. Mismo resultado, cinco veces el costo. |

La 5 tiene una ventaja que nos regala la infraestructura: con Ollama compartido
por LAN, la latencia es compartida y visible. Cuando el sistema multiagente
dispara cinco llamadas en lugar de una, **todo el salón lo siente**. La demo del
costo escondido se vuelve física.

Después de la 4: `scripts/<tu-sistema>/reiniciar-datos`.

---

## Cierre

**Evaluación.** Los casos de prueba de [`evals/`](../evals/). Un agente se
verifica por su **efecto**, no comparando cadenas — porque la temperatura hace
que la misma entrada dé salidas distintas. Y lo que un agente **no** debe hacer
es más difícil de verificar, y más caro de equivocar, que lo que debe hacer.

**Cuándo agentes y cuándo no.** Si los pasos son fijos y conocidos, escribe la
función. Un agente sirve cuando no sabes de antemano cuántos pasos hacen falta
ni en qué orden. Nada más.

**Hacia dónde seguir.** Lo que no hicimos y por qué:

- **MCP de verdad** — sacar las tools a un servidor y que las use
  cualquier cliente. Es el siguiente paso natural de lo que escribimos hoy.
- **Frameworks.** Ahora que viste el bucle, LangGraph o los Agents SDK son una
  conveniencia, no una caja negra. Esa diferencia era el objetivo del taller.
- **Modelos grandes.** Todo lo que vimos corrió en un 9B local. Con un modelo
  frontera, varias de las fallas del bloque 5 dejan de aparecer — y las
  restantes son de diseño, no de modelo. Esas no las arregla ningún modelo.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import importlib

# Recargar agentlib ANTES que lo que depende de el. reload() NO es recursivo:
# si agentlib quedo cacheado de una version anterior, recargar el modulo de
# arriba falla con un "ImportError: cannot import name ... from 'agentlib'"
# que parece del modulo de arriba y en realidad es del paquete de abajo.
#
# Sin esto, basta con que alguien actualice el repo con el kernel abierto para
# que la primera celda falle con un error que no menciona la causa.
import agentlib
import agentlib.ollama
import agentlib.traza
for _modulo in (agentlib.ollama, agentlib.traza, agentlib):
    importlib.reload(_modulo)

# El .env de la raiz entra al entorno AQUI, antes de la primera llamada. Un
# kernel de Jupyter puede traer valores viejos -VS Code inyecta el .env al
# arrancarlo- y reload() no limpia variables de entorno. cargar_env() si, y
# avisa de las claves que borro por ya no estar en el archivo.
from agentlib import cargar_env

_env, _claves, _viejas = cargar_env()
print(f"  .env   : {_env}  ({_claves} claves)")
if _viejas:
    print(f"           quitadas del entorno, ya no estan en el .env: {', '.join(_viejas)}")

from agentlib import chat, normalizar_argumentos, registrar

# TODO: pendiente del caso de estudio (planeacion/caso-de-estudio.md)